# 03 — ResNet50 Transfer Learning (PyTorch)
**Objectif** : ResNet50 pré-entraîné ImageNet en 2 phases (base gelée → fine-tuning).
PyTorch : utilisation de `torchvision.models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)`.


In [8]:
import sys; sys.path.append('../../src')
import torch, torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import classification_report
from data_loader import get_dataloaders, compute_class_weights
from models import ResNet50Transfer
from utils import plot_learning_curves, plot_confusion_matrix, print_model_summary
import warnings; warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASSES = ['NORMAL', 'BACTERIA', 'VIRUS']
print(f"Device : {DEVICE}"); torch.manual_seed(42)


Device : cpu


In [9]:
# ── Données avec augmentation ─────────────────────────────────
train_loader, val_loader, test_loader, train_ds, _, _ = get_dataloaders(
    '../../data/chest_xray/train',
    '../../data/chest_xray/val',
    '../../data/chest_xray/test',
    augment=True, batch_size=32)
cw = compute_class_weights(train_ds, DEVICE)
criterion = nn.CrossEntropyLoss(weight=cw)


Classes : {'NORMAL': 0, 'PNEUMONIA': 1}
Train   : 5216 | Val: 16 | Test: 624


In [10]:
# ── Helper : boucle d'entraînement ───────────────────────────
def run_phase(model, train_loader, val_loader, criterion, lr, epochs,
              save_path, patience=6):
    optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scheduler = ReduceLROnPlateau(optimizer, factor=0.5, patience=3)
    history   = {'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[]}
    best_acc  = 0.; no_imp = 0

    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        tr_l, tr_c, tr_n = 0., 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward(); optimizer.step()
            tr_l += loss.item()*imgs.size(0)
            tr_c += (model(imgs).argmax(1) == labels).sum().item()
            tr_n += imgs.size(0)
        # Val
        model.eval()
        vl_l, vl_c, vl_n = 0., 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                out = model(imgs)
                vl_l += criterion(out, labels).item()*imgs.size(0)
                vl_c += (out.argmax(1)==labels).sum().item()
                vl_n += imgs.size(0)
        tr_acc = tr_c/tr_n; vl_acc = vl_c/vl_n
        scheduler.step(vl_l/vl_n)
        history['train_loss'].append(tr_l/tr_n); history['val_loss'].append(vl_l/vl_n)
        history['train_acc'].append(tr_acc);      history['val_acc'].append(vl_acc)
        print(f"Ep {epoch:2d}  train={tr_acc:.4f}  val={vl_acc:.4f}")
        if vl_acc > best_acc:
            best_acc = vl_acc; no_imp = 0
            torch.save({'epoch':epoch,'model_state':model.state_dict(),'val_acc':vl_acc}, save_path)
        else:
            no_imp += 1
            if no_imp >= patience: print(f"EarlyStopping ép. {epoch}"); break
    print(f"Meilleure val_accuracy : {best_acc:.4f}")
    return history, best_acc


In [11]:
# ── Phase 1 : base gelée (LR = 1e-4) ────────────────────────
print("="*55 + "\nPhase 1 — Base ResNet50 gelée\n" + "="*55)
model_p1 = ResNet50Transfer(unfreeze_top=0).to(DEVICE)
print_model_summary(model_p1, 'ResNet50 Phase 1')

history_p1, best_p1 = run_phase(
    model_p1, train_loader, val_loader, criterion,
    lr=1e-4, epochs=15,
    save_path='../../models/resnet50_phase1.pt')


Phase 1 — Base ResNet50 gelée

Modèle : ResNet50 Phase 1
  Total paramètres      :   24,690,755
  Entraînables          :    1,182,723  (4.8%)
  Gelés                 :   23,508,032  (95.2%)



RuntimeError: weight tensor should be defined either for all 3 classes or no classes but got weight tensor of shape: [2]

In [ ]:
# ── Phase 2 : fine-tuning top-3 layer groups (LR = 1e-5) ────
print("="*55 + "\nPhase 2 — Fine-tuning (LR=1e-5)\n" + "="*55)
model_p2 = ResNet50Transfer(unfreeze_top=3).to(DEVICE)
# Charger les poids Phase 1
ckpt = torch.load('../../models/resnet50_phase1.pt', map_location=DEVICE)
model_p2.load_state_dict(ckpt['model_state'], strict=False)
print_model_summary(model_p2, 'ResNet50 Phase 2')

history_p2, best_p2 = run_phase(
    model_p2, train_loader, val_loader, criterion,
    lr=1e-5, epochs=25,
    save_path='../../models/resnet50_best.pt')

print(f"\nGain fine-tuning : +{(best_p2-best_p1)*100:.1f}%")


In [ ]:
# ── Courbes combinées Phase 1 + Phase 2 ──────────────────────
p1_len = len(history_p1['train_acc'])
combined = {k: history_p1[k] + history_p2[k]
            for k in ['train_loss','val_loss','train_acc','val_acc']}
plot_learning_curves(combined, 'ResNet50 Transfer Learning', phase_break=p1_len+1)


In [ ]:
# ── Évaluation finale ─────────────────────────────────────────
ckpt = torch.load('../../models/resnet50_best.pt', map_location=DEVICE)
model_p2.load_state_dict(ckpt['model_state'])
model_p2.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        out = model_p2(imgs.to(DEVICE))
        all_preds.extend(out.argmax(1).cpu().tolist())
        all_labels.extend(labels.tolist())
print(classification_report(all_labels, all_preds, target_names=CLASSES))
plot_confusion_matrix(all_labels, all_preds)
